# <center>**Phân tích tập phổ biến từ dữ liệu mua hàng tại siêu thị bằng thuật toán Apriori**<center>

Sử dụng thư viện `apyori` để tìm các luật kết hợp từ tập dữ liệu `basket_optimisation.csv`.

Do dữ liệu này không có tiêu đề cột (header) và mỗi dòng đã là một giao dịch (transaction) với các mặt hàng phân tách, chúng ta cần đọc nó vào một danh sách các danh sách (list of lists) để chuẩn bị cho thuật toán Apriori.

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
# Cài đặt thư viện nếu chưa có
!pip install apyori

import pandas as pd
from apyori import apriori

# Tên file dữ liệu đã tải lên
FILE_NAME = "/basket_optimisation.csv"

# Đọc dữ liệu
# Đọc file CSV, không có header, xử lý các giá trị thiếu (NaN)
dataset = pd.read_csv(FILE_NAME, header=None)

In [8]:
# Chuyển dữ liệu sang định dạng List of Lists (transactions)
# Thư viện apyori yêu cầu input là một danh sách chứa tất cả các giao dịch,
# trong đó mỗi giao dịch là một danh sách các mặt hàng đã mua.
transactions = []
# Số hàng (giao dịch) trong tập dữ liệu
num_rows = len(dataset)
# Số cột tối đa (số mặt hàng tối đa trong một giao dịch)
num_cols = len(dataset.columns)

for i in range(0, num_rows):
    # Lọc các giá trị NaN/None, chuyển đổi thành chuỗi và thêm vào danh sách giao dịch
    transaction = [str(dataset.values[i, j]) for j in range(0, num_cols) if pd.notna(dataset.values[i, j])]
    if transaction: # Đảm bảo giao dịch không rỗng
        transactions.append(transaction)

print(f"Tổng số giao dịch: {len(transactions)}")
print("Ví dụ 3 giao dịch đầu tiên:")
print(transactions[:3])

Tổng số giao dịch: 7501
Ví dụ 3 giao dịch đầu tiên:
[['shrimp', 'almonds', 'avocado', 'vegetables mix', 'green grapes', 'whole weat flour', 'yams', 'cottage cheese', 'energy drink', 'tomato juice', 'low fat yogurt', 'green tea', 'honey', 'salad', 'mineral water', 'salmon', 'antioxydant juice', 'frozen smoothie', 'spinach', 'olive oil'], ['burgers', 'meatballs', 'eggs'], ['chutney']]


## **Áp dụng giải thuật Apriori**

Chúng ta áp dụng hàm `apriori` với các tham số quan trọng để khai thác các luật kết hợp (Association Rules):

* **min_support**: Ngưỡng Support tối thiểu. Nếu một mặt hàng/tập hợp mặt hàng xuất hiện ít hơn ngưỡng này, nó sẽ bị loại. Giả sử 3 lần/ngày * 7 ngày / 7500 giao dịch ≈ 0.0028. Ta đặt **0.003**.
* **min_confidence**: Ngưỡng Confidence tối thiểu. Ta đặt **0.2** (20%).
* **min_lift**: Ngưỡng Lift tối thiểu. Ta đặt **3.0** để tìm kiếm các mối quan hệ mạnh.
* **min_length**: Độ dài tối thiểu của itemset (thường là 2).

In [9]:
# Áp dụng giải thuật Apriori
# Đặt ngưỡng min_support = 0.003 (khoảng 21 lần/tuần trên tổng số 7501 giao dịch)
rules = apriori(transactions=transactions,
                min_support=0.003,
                min_confidence=0.2,
                min_lift=3,
                min_length=2,
                max_length=3) # Giới hạn 3 mặt hàng để tăng tốc độ tính toán

# Chuyển kết quả sang list để dễ dàng xử lý
results = list(rules)

print(f"Số lượng luật kết hợp tìm thấy: {len(results)}")

Số lượng luật kết hợp tìm thấy: 56


## **Trích xuất và phân tích kết quả**

Hàm `apriori` trả về một đối tượng chứa các **RelationalRecord**. Chúng ta cần định dạng lại kết quả để hiển thị rõ ràng các luật **(Antecedents -> Consequents)** cùng với các chỉ số **Support**, **Confidence**, và **Lift**.

In [10]:
# Hàm trợ giúp để định dạng kết quả thành DataFrame
def inspect(results):
    rhses = [] # Consequents (Vế sau)
    lhses = [] # Antecedents (Vế trước)
    supports = []
    confidences = []
    lifts = []

    for result in results:
        # Lấy thông tin luật mạnh nhất từ mỗi tập phổ biến
        for rule in result.ordered_statistics:
            # Chỉ lấy các luật có Antecedents và Consequents khác rỗng
            if rule.items_base and rule.items_add:
                # Antecedents (Phần trước)
                lhses.append(tuple(rule.items_base))
                # Consequents (Phần sau)
                rhses.append(tuple(rule.items_add))
                # Support của itemset (tần suất xuất hiện toàn bộ)
                supports.append(result.support)
                # Confidence của luật (xác suất mua vế sau khi đã mua vế trước)
                confidences.append(rule.confidence)
                # Lift (độ mạnh của mối quan hệ, > 1 là mối quan hệ tích cực)
                lifts.append(rule.lift)

    return pd.DataFrame({
        'Antecedents': lhses,
        'Consequents': rhses,
        'Support': supports,
        'Confidence': confidences,
        'Lift': lifts
    })

# Chuyển đổi và hiển thị 10 luật có chỉ số Lift cao nhất
df_results = inspect(results)

print("--- 10 Luật Kết hợp có chỉ số Lift cao nhất (Mối quan hệ mạnh nhất) ---")
df_results.sort_values(by='Lift', ascending=False).head(10)

--- 10 Luật Kết hợp có chỉ số Lift cao nhất (Mối quan hệ mạnh nhất) ---


,Antecedents,Consequents,Support,Confidence,Lift
69,"(mineral water, whole wheat pasta)","(olive oil,)",0.003866,0.402778,6.115863
56,"(tomato sauce,)","(spaghetti, ground beef)",0.003066,0.216981,5.535971
3,"(fromage blanc,)","(honey,)",0.003333,0.245098,5.164271
58,"(spaghetti, tomato sauce)","(ground beef,)",0.003066,0.489362,4.980600
0,"(light cream,)","(chicken,)",0.004533,0.290598,4.843951
2,"(pasta,)","(escalope,)",0.005866,0.372881,4.700812
32,"(french fries, herb & pepper)","(ground beef,)",0.003200,0.461538,4.697422
16,"(spaghetti, cereals)","(ground beef,)",0.003066,0.460000,4.681764
31,"(french fries, ground beef)","(herb & pepper,)",0.003200,0.230769,4.665768
8,"(pasta,)","(shrimp,)",0.005066,0.322034,4.506672


---
# <Center>**END**<center>